# PAS-RAG Project Data Exploration

This notebook audits the current project files before experimentation.
Each code cell is followed by a markdown cell that explains findings in plain language.


In [1]:
from pathlib import Path
from collections import Counter
import json
import statistics

BASE = Path.cwd()
print('Working directory:', BASE)

Working directory: /Users/shayan/Projects/PAS_RAG


In [2]:
target_files = [
    'ReadMe.txt',
    'NYC_SYNTHETIC_GENERATION_README.md',
    'expanded_anchor_registry.jsonl',
    'expanded_chunks.jsonl',
    'expanded_eval_queries.jsonl',
    'pas_rag_end_to_end_patched.py',
]

inventory = []
for name in target_files:
    path = BASE / name
    if not path.exists():
        inventory.append((name, 'MISSING', None, None))
        continue
    size_bytes = path.stat().st_size
    line_count = sum(1 for _ in path.open('r', encoding='utf-8'))
    inventory.append((name, 'OK', size_bytes, line_count))

for row in inventory:
    print(row)


('ReadMe.txt', 'OK', 242, 5)
('NYC_SYNTHETIC_GENERATION_README.md', 'OK', 668, 29)
('expanded_anchor_registry.jsonl', 'OK', 16406, 30)
('expanded_chunks.jsonl', 'OK', 2964840, 1010)
('expanded_eval_queries.jsonl', 'OK', 297278, 423)
('pas_rag_end_to_end_patched.py', 'OK', 22804, 578)


Expected inventory (current snapshot):
- `expanded_anchor_registry.jsonl`: 30 lines (30 anchors)
- `expanded_chunks.jsonl`: 1010 lines (1010 chunks)
- `expanded_eval_queries.jsonl`: 423 lines (423 eval queries)
- `pas_rag_end_to_end_patched.py`: 563 lines

Interpretation: the project is compact and centered around one pipeline script plus three main JSONL datasets.


In [3]:
def load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

anchors = load_jsonl(BASE / 'expanded_anchor_registry.jsonl')
chunks = load_jsonl(BASE / 'expanded_chunks.jsonl')
queries = load_jsonl(BASE / 'expanded_eval_queries.jsonl')

print('anchors:', len(anchors))
print('chunks:', len(chunks))
print('queries:', len(queries))

anchors: 30
chunks: 1010
queries: 423


Loaded dataset sizes (current snapshot): **30 anchors**, **1010 chunks**, **423 queries**.
This matches the synthetic README counts, so files appear internally consistent at the top level.


In [4]:
anchor_key_counts = Counter()
for a in anchors:
    anchor_key_counts.update(a.keys())

anchor_types = Counter(a.get('anchor_type') for a in anchors)
anchor_boroughs = Counter(a.get('borough') for a in anchors)

print('Anchor top-level key presence:', dict(anchor_key_counts))
print('Anchor types:', dict(anchor_types))
print('Anchor boroughs:', dict(anchor_boroughs))
print('Sample anchor record:', json.dumps(anchors[0], indent=2)[:900])

Anchor top-level key presence: {'anchor_id': 30, 'name': 30, 'canonical_name': 30, 'anchor_type': 30, 'geo': 30, 'borough': 30, 'coverage_summary': 30, 'pas_enabled': 30}
Anchor types: {'landmark': 6, 'street_address': 8, 'transit_hub': 7, 'park': 4, 'library': 1, 'market': 1, 'museum': 1, 'stadium': 1, 'zoo': 1}
Anchor boroughs: {'Brooklyn': 10, 'Manhattan': 12, 'Queens': 4, 'Bronx': 3, 'Staten Island': 1}
Sample anchor record: {
  "anchor_id": "A_BoroHall",
  "name": "Brooklyn Borough Hall",
  "canonical_name": "Brooklyn Borough Hall",
  "anchor_type": "landmark",
  "geo": {
    "lat": 40.69263,
    "lon": -73.99054
  },
  "borough": "Brooklyn",
  "coverage_summary": {
    "linked_chunk_count": 229,
    "linked_categories": [
      "bar",
      "bookstore",
      "cafe",
      "grocery_store",
      "hospital",
      "hotel",
      "library",
      "museum",
      "park",
      "pharmacy",
      "restaurant",
      "subway_station",
      "theater"
    ],
    "linked_neighborhoods": 

Anchor file structure:
- Uniform schema across all 30 records: `anchor_id`, `name`, `canonical_name`, `anchor_type`, `geo`, `borough`, `coverage_summary`, `pas_enabled`
- Type mix includes landmarks, transit hubs, and addresses (`street_address` is largest group).
- Borough spread is NYC-wide but Manhattan/Brooklyn dominate.

Project relation: this file is the PAS public-anchor universe used for privacy-preserving substitution.


In [5]:
chunk_key_counts = Counter()
metadata_key_counts = Counter()
spatial_key_counts = Counter()
provenance_key_counts = Counter()
embedding_key_counts = Counter()

for c in chunks:
    chunk_key_counts.update(c.keys())
    metadata_key_counts.update((c.get('metadata') or {}).keys())
    spatial_key_counts.update((c.get('spatial') or {}).keys())
    provenance_key_counts.update((c.get('provenance') or {}).keys())
    embedding_key_counts.update((c.get('embedding') or {}).keys())

print('Chunk top-level keys:', dict(chunk_key_counts))
print('Metadata keys (top 20):', dict(metadata_key_counts.most_common(20)))
print('Spatial keys:', dict(spatial_key_counts))
print('Provenance keys:', dict(provenance_key_counts))
print('Embedding keys:', dict(embedding_key_counts))
print('Sample chunk record (truncated):', json.dumps(chunks[0], indent=2)[:1300])

Chunk top-level keys: {'chunk_id': 1010, 'doc_id': 1010, 'source_id': 1010, 'title': 1010, 'category': 1010, 'subcategory': 1010, 'metadata': 1010, 'content': 1010, 'supporting_facts': 1010, 'spatial': 1010, 'provenance': 1010, 'embedding': 1010}
Metadata keys (top 20): {'item_id': 1010, 'item_type': 1010, 'query_group': 1010, 'address': 1010, 'neighborhood': 1010, 'borough': 1010, 'geo': 1010, 'tags': 1010, 'hours': 678, 'amenities': 516, 'price_range': 271, 'cuisine': 183, 'signature_items': 183, 'services': 140, 'lines_served': 124, 'station_features': 120, 'specialties': 120, 'beverage_focus': 110, 'size_class': 95, 'check_in': 88}
Spatial keys: {'anchor_tags': 1010, 'primary_anchor_id': 1010, 'primary_anchor_name': 1010, 'pas_ready': 1010}
Provenance keys: {'source_file': 1010, 'anchor_tag_file': 1010, 'source_type': 1010, 'source_name': 1010, 'conversion_note': 1010, 'last_updated': 1010}
Embedding keys: {'model': 1010, 'vector': 1010}
Sample chunk record (truncated): {
  "chunk_

Chunk file is rich and regular:
- All 1010 chunks share the same top-level schema (document identity, text, metadata, spatial tags, provenance, embedding).
- `metadata` is category-specific (e.g., `cuisine`, `lines_served`, `services` appear only when relevant).
- `spatial.anchor_tags` and `spatial.primary_anchor_id` directly support PAS retrieval.
- `provenance.source_type` distinguishes mostly `synthetic_expansion` plus a small `synthetic_demo` subset.


In [6]:
category_counts = Counter(c.get('category') for c in chunks)
borough_counts = Counter(c.get('metadata', {}).get('borough') for c in chunks)
query_group_counts = Counter(c.get('metadata', {}).get('query_group') for c in chunks)
embedding_dims = Counter(len(c.get('embedding', {}).get('vector', [])) for c in chunks)
content_lengths = [len(c.get('content', '')) for c in chunks]
source_types = Counter(c.get('provenance', {}).get('source_type') for c in chunks)

print('Category counts:', dict(category_counts))
print('Chunk borough counts:', dict(borough_counts))
print('query_group counts:', dict(query_group_counts))
print('Embedding dimension distribution:', dict(embedding_dims))
print('Content length chars avg/min/max:', round(statistics.mean(content_lengths), 1), min(content_lengths), max(content_lengths))
print('Provenance source_type:', dict(source_types))

Category counts: {'restaurant': 183, 'hotel': 88, 'subway_station': 124, 'cafe': 110, 'museum': 55, 'park': 95, 'library': 45, 'pharmacy': 70, 'grocery_store': 80, 'theater': 40, 'bookstore': 55, 'hospital': 25, 'bar': 40}
Chunk borough counts: {'Brooklyn': 305, 'Manhattan': 314, 'Bronx': 111, 'Queens': 248, 'Staten Island': 32}
query_group counts: {'restaurant': 183, 'hotel': 88, 'subway station': 4, 'subway_station': 120, 'cafe': 110, 'museum': 55, 'park': 95, 'library': 45, 'pharmacy': 70, 'grocery_store': 80, 'theater': 40, 'bookstore': 55, 'hospital': 25, 'bar': 40}
Embedding dimension distribution: {8: 1010}
Content length chars avg/min/max: 193.6 140 254
Provenance source_type: {'synthetic_demo': 10, 'synthetic_expansion': 1000}


Observed content profile:
- Category distribution is intentionally broad (13 categories), with restaurants largest (183).
- Borough distribution covers all five boroughs; Manhattan and Brooklyn are largest.
- Embeddings are fixed-length 8D vectors under `mock-mini-embedding`, indicating synthetic placeholders (not production embeddings).
- `query_group` mostly mirrors `category`, with a small normalization inconsistency (`subway_station` vs `subway station`).


In [7]:
anchor_tags_per_chunk = Counter(len(c.get('spatial', {}).get('anchor_tags', [])) for c in chunks)
direction_bins = Counter()
distance_bins = Counter()
primary_anchor_counts = Counter(c.get('spatial', {}).get('primary_anchor_id') for c in chunks)

for c in chunks:
    for t in c.get('spatial', {}).get('anchor_tags', []):
        direction_bins[t.get('direction_bin')] += 1
        distance_bins[t.get('distance_bin')] += 1

print('Anchor tags per chunk:', dict(anchor_tags_per_chunk))
print('Direction bin distribution:', dict(direction_bins))
print('Distance bin distribution:', dict(distance_bins))
print('Top 10 primary anchors:', primary_anchor_counts.most_common(10))

Anchor tags per chunk: {3: 10, 4: 1000}
Direction bin distribution: {'N': 571, 'SE': 464, 'E': 716, 'NW': 398, 'SW': 496, 'NE': 536, 'W': 435, 'S': 414}
Distance bin distribution: {'0-0.5mi': 618, '2mi+': 1072, '0.5-1mi': 1039, '1-2mi': 1301}
Top 10 primary anchors: [('A_QueensPlaza', 81), ('A_MetMuseum', 76), ('A_FlushingMain', 73), ('A_FordhamRd', 67), ('A_BoroHall', 56), ('A_Bushwick100', 51), ('A_BedfordAve', 48), ('A_FortGreenePark', 48), ('A_ChelseaMarket', 46), ('A_AstoriaPark', 43)]


Spatial annotation findings:
- Most chunks have **4 anchor tags**; a small subset has 3.
- Direction bins cover all 8 sectors and distance bins cover all PAS rings (`0-0.5mi`, `0.5-1mi`, `1-2mi`, `2mi+`).
- This confirms the corpus is already indexed into PAS-compatible discrete spatial tokens.


In [8]:
query_key_counts = Counter()
semantic_key_counts = Counter()
spatial_key_counts_q = Counter()
token_key_counts = Counter()
dir_bins_q = Counter()
dist_bins_q = Counter()
gt_lengths = Counter()

for q in queries:
    query_key_counts.update(q.keys())
    semantic_key_counts.update((q.get('semantic_intent') or {}).keys())
    spatial_key_counts_q.update((q.get('spatial_intent') or {}).keys())
    token_key_counts.update((q.get('pas_token_template') or {}).keys())
    dir_bins_q[q.get('pas_token_template', {}).get('direction_bin')] += 1
    dist_bins_q[q.get('pas_token_template', {}).get('distance_bin')] += 1
    gt_lengths[len(q.get('ground_truth_doc_ids', []))] += 1

print('Query top-level keys:', dict(query_key_counts))
print('semantic_intent keys:', dict(semantic_key_counts))
print('spatial_intent keys:', dict(spatial_key_counts_q))
print('pas_token_template keys:', dict(token_key_counts))
print('PAS direction bins in queries:', dict(dir_bins_q))
print('PAS distance bins in queries:', dict(dist_bins_q))
print('Ground-truth list lengths:', dict(gt_lengths))
print('Sample query (truncated):', json.dumps(queries[-1], indent=2)[:1000])

Query top-level keys: {'qid': 423, 'raw_query': 423, 'semantic_intent': 423, 'spatial_intent': 423, 'pas_token_template': 423, 'ground_truth_doc_ids': 423, 'notes': 423}
semantic_intent keys: {'entity_type': 423, 'must_have_tags': 423, 'attributes': 423}
spatial_intent keys: {'anchor_hint': 423, 'direction_constraint': 423, 'radius_miles': 423, 'user_location': 423}
pas_token_template keys: {'anchor_id': 423, 'anchor_name': 423, 'direction_bin': 423, 'distance_bin': 423}
PAS direction bins in queries: {'N': 43, 'ANY': 144, 'NE': 33, 'W': 34, 'SE': 31, 'E': 35, 'SW': 37, 'NW': 33, 'S': 33}
PAS distance bins in queries: {'0.5-1mi': 145, '1-2mi': 208, '0-0.5mi': 70}
Ground-truth list lengths: {3: 34, 1: 206, 4: 99, 2: 79, 5: 5}
Sample query (truncated): {
  "qid": "Q_SYN_2451",
  "raw_query": "Find with lab services hospitals se of Bryant Park within 2 miles",
  "semantic_intent": {
    "entity_type": "hospital",
    "must_have_tags": [
      "hospital"
    ],
    "attributes": [
      {


Evaluation query structure is consistent across all 423 records.
Each query includes:
- semantic intent
- spatial intent (including true user location in this synthetic benchmark)
- PAS token template
- ground-truth doc IDs

Ground-truth list length varies from 1 to 5, enabling ranking metrics like recall@k and MRR.


In [9]:
anchor_ids = {a['anchor_id'] for a in anchors}
doc_ids = {c['doc_id'] for c in chunks}

missing_anchor_refs = 0
for c in chunks:
    for t in c.get('spatial', {}).get('anchor_tags', []):
        if t.get('anchor_id') not in anchor_ids:
            missing_anchor_refs += 1

missing_gt_doc_refs = 0
for q in queries:
    for d in q.get('ground_truth_doc_ids', []):
        if d not in doc_ids:
            missing_gt_doc_refs += 1

print('Missing anchor references from chunk tags:', missing_anchor_refs)
print('Missing ground-truth doc references from queries:', missing_gt_doc_refs)

Missing anchor references from chunk tags: 0
Missing ground-truth doc references from queries: 0


Referential integrity check: expected result is **0 missing references** for both anchors and ground-truth doc IDs.
If non-zero appears here, downstream retrieval/evaluation experiments may silently fail or under-report scores.


In [10]:
pipeline_text = (BASE / 'pas_rag_end_to_end_patched.py').read_text(encoding='utf-8')
expected_files = ['chunks.jsonl', 'anchor_registry.jsonl', 'eval_queries.json', 'eval_queries.jsonl']
for name in expected_files:
    print(name, 'referenced_in_script=', name in pipeline_text, 'exists_in_folder=', (BASE / name).exists())

chunks.jsonl referenced_in_script= True exists_in_folder= False
anchor_registry.jsonl referenced_in_script= True exists_in_folder= False
eval_queries.json referenced_in_script= True exists_in_folder= False
eval_queries.jsonl referenced_in_script= True exists_in_folder= False


Important compatibility note:
- The pipeline script references base names (`chunks.jsonl`, `anchor_registry.jsonl`, `eval_queries*.json*`).
- This folder currently stores `expanded_*` filenames.

Before running the script, either rename/copy files or patch load paths to the expanded filenames.


## Bottom-line Summary
- The dataset is highly likely synthetic/AI-generated (explicit synthetic provenance fields + mock embeddings + synthetic readme).
- Data schema is consistent and experiment-ready for PAS-style retrieval.
- The key immediate fix before experiments is filename alignment between data files and the pipeline script.

After this audit, you can proceed to retrieval experiments (baseline vs PAS) with confidence in data integrity.
